In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "catcher-welfare-ragas-eval"
os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY")
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [ ]:
from pathlib import Path
import sys

def find_project_root() -> Path:
    curr = Path.cwd()
    for parent in [curr] + list(curr.parents):
        if (parent / "pyproject.toml").exists():
            return parent
    return curr

PROJECT_ROOT = find_project_root()
src_dir = str(PROJECT_ROOT / "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

print("PROJECT_ROOT:", PROJECT_ROOT)

# 1. 문서 로드 — 2026_hope_ladder_selected.pdf

In [ ]:
import logging
logging.getLogger("pdfminer").setLevel(logging.ERROR)

from langchain_community.document_loaders import PDFPlumberLoader

PDF_PATH = PROJECT_ROOT / "data/raw/pdf/welfare/2026_hope_ladder_selected.pdf"

loader = PDFPlumberLoader(str(PDF_PATH))
all_docs = loader.load()

for d in all_docs:
    d.metadata["source"] = PDF_PATH.name

print(f"총 페이지 수: {len(all_docs)}")
print(all_docs[0].page_content[:300])

# 2. 문서 split

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=120
)

split_docs = text_splitter.split_documents(all_docs)
print(f"청킹 후 문서 수: {len(split_docs)}")

# 3. 임베딩 + 벡터 DB

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from ragas.embeddings import LangchainEmbeddingsWrapper

base_embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
ragas_embeddings = LangchainEmbeddingsWrapper(base_embeddings)

vectorstore = FAISS.from_documents(split_docs, base_embeddings)
print(f"벡터 수: {vectorstore.index.ntotal}")

# 4. Retriever + RAG 함수

In [ ]:
from langchain_openai import ChatOpenAI

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, max_tokens=300)

def run_rag(q: str):
    docs = retriever.invoke(q)
    context_texts = [doc.page_content for doc in docs]
    answer = llm.invoke(
        f"""질문: {q}

아래 문서에 있는 내용만 사용해서 핵심 답변을 2문장 이내로 작성하세요.
문서에 없는 내용은 절대 추가하지 마세요.

문서:
{context_texts}"""
    ).content
    sources = [doc.metadata.get("source", "") for doc in docs]
    return answer, context_texts, sources

In [ ]:
# 단건 테스트
answer, contexts, sources = run_rag("청년내일저축계좌의 정부 매칭 한도는 얼마인가?")
print("답변:", answer)
print("출처:", sources[0])

In [ ]:
def target(inputs: dict):
    q = inputs["question"]
    answer, contexts, sources = run_rag(q)
    return {
        "answer": answer,
        "contexts": contexts,
    }

# 5. LangSmith Dataset 생성

In [ ]:
from langsmith import Client

client = Client()
dataset_name = "catcher-welfare-ragas-eval"

questions = [
    "여성청소년 생리용품 지원의 월 지원금은 얼마인가?",
    "임신 사전건강관리 지원사업에서 여성에게 지원하는 최대 금액은 얼마인가?",
    "저소득 청소년부모 아동양육비 지원의 월 지원금은 얼마인가?",
    "3~5세 유치원 학비 중 국·공립유치원 교육비는 월 얼마인가?",
    "청년내일저축계좌의 정부 매칭 한도는 얼마인가?",
]

ground_truths = [
    "여성청소년 생리용품 지원의 월 지원금은 1만 4,000원이다.",
    "임신 사전건강관리 지원사업에서 여성에게 지원하는 최대 금액은 13만 원이다.",
    "저소득 청소년부모 아동양육비 지원의 월 지원금은 25만 원이다.",
    "3~5세 유치원 학비 중 국·공립유치원 교육비는 월 10만 원이다.",
    "청년내일저축계좌의 정부 매칭 한도는 월 최대 30만 원이며, 3년 만기 시 최대 1,440만 원 적립 가능하다.",
]

existing = [d for d in client.list_datasets() if d.name == dataset_name]
if existing:
    dataset = existing[0]
    print(f"기존 dataset 사용: {dataset.name}")
else:
    dataset = client.create_dataset(
        dataset_name=dataset_name,
        description="2026 희망사다리 복지정책 RAG — RAGAS 지표 평가"
    )
    for q, gt in zip(questions, ground_truths):
        client.create_example(
            inputs={"question": q},
            outputs={"ground_truth": gt},
            dataset_id=dataset.id
        )
    print(f"새 dataset 생성: {dataset.name} ({len(questions)}개)")

# 6. Evaluator 정의

| # | Evaluator | 유형 | 설명 |
|---|---|---|---|
| 1 | `faithfulness` | RAGAS | 답변이 검색된 문서에만 근거하는가 |
| 2 | `answer_relevancy` | RAGAS | 답변이 질문에 얼마나 관련 있는가 |
| 3 | `context_precision` | RAGAS | 검색된 문서 중 실제로 유용한 비율 |
| 4 | `context_recall` | RAGAS | 정답에 필요한 정보가 검색됐는가 |
| 5 | `answer_correctness` | RAGAS | 답변이 정답과 얼마나 일치하는가 |
| 6 | `contains_amount` | Heuristic | 구체적 금액(숫자+원)이 포함됐는가 |
| 7 | `no_hallucination` | LLM judge | 문서에 없는 정보를 생성하지 않았는가 |

In [ ]:
import re
from datasets import Dataset
from ragas import evaluate as ragas_evaluate
from ragas.metrics import (
    Faithfulness,
    AnswerRelevancy,
    ContextPrecision,
    ContextRecall,
    AnswerCorrectness,
)
from langchain_openai import ChatOpenAI

judge_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)


def _ragas_score(metric, question, answer, contexts, ground_truth):
    """RAGAS 단일 샘플 평가 헬퍼"""
    ds = Dataset.from_dict({
        "question":     [question],
        "answer":       [answer],
        "contexts":     [contexts],
        "ground_truth": [ground_truth],
    })
    result = ragas_evaluate(dataset=ds, metrics=[metric])
    key = list(result.scores[0].keys())[0]
    val = result.scores[0][key]
    return float(val) if val is not None else 0.0


# ── 필수 4개 (RAGAS) ────────────────────────────────────────

def faithfulness_evaluator(run, example):
    score = _ragas_score(
        Faithfulness(),
        example.inputs["question"],
        run.outputs["answer"],
        run.outputs["contexts"],
        example.outputs["ground_truth"],
    )
    return {"key": "faithfulness", "score": score}


def answer_relevancy_evaluator(run, example):
    score = _ragas_score(
        AnswerRelevancy(),
        example.inputs["question"],
        run.outputs["answer"],
        run.outputs["contexts"],
        example.outputs["ground_truth"],
    )
    return {"key": "answer_relevancy", "score": score}


def context_precision_evaluator(run, example):
    score = _ragas_score(
        ContextPrecision(),
        example.inputs["question"],
        run.outputs["answer"],
        run.outputs["contexts"],
        example.outputs["ground_truth"],
    )
    return {"key": "context_precision", "score": score}


def context_recall_evaluator(run, example):
    score = _ragas_score(
        ContextRecall(),
        example.inputs["question"],
        run.outputs["answer"],
        run.outputs["contexts"],
        example.outputs["ground_truth"],
    )
    return {"key": "context_recall", "score": score}


# ── 추가 3개 ────────────────────────────────────────────────

def answer_correctness_evaluator(run, example):
    """RAGAS — 정답과 의미적 일치도 (0~1)"""
    score = _ragas_score(
        AnswerCorrectness(),
        example.inputs["question"],
        run.outputs["answer"],
        run.outputs["contexts"],
        example.outputs["ground_truth"],
    )
    return {"key": "answer_correctness", "score": score}


def contains_amount_evaluator(run, example):
    """Heuristic — 구체적 금액(숫자+원)이 포함됐는가 (0 or 1)"""
    answer = run.outputs.get("answer", "")
    has_amount = bool(re.search(r'\d[\d,]*\s*(원|만원|만\s*원)', answer))
    return {"key": "contains_amount", "score": 1 if has_amount else 0}


def no_hallucination_evaluator(run, example):
    """LLM judge — 문서에 없는 정보를 생성하지 않았는가 (0~1)"""
    answer = run.outputs.get("answer", "")
    contexts = run.outputs.get("contexts", [])
    prompt = f"""아래 답변이 문서에 있는 내용만 사용했는지 평가해줘.
문서에 없는 정보(수치, 정책명, 조건 등)를 추가했으면 낮게,
문서 내용만 사용했으면 높게. 0~1 숫자 하나만 출력해.

문서:
{contexts}

답변: {answer}"""
    score = judge_llm.invoke(prompt).content.strip()
    try:
        return {"key": "no_hallucination", "score": float(score)}
    except ValueError:
        return {"key": "no_hallucination", "score": 0.0}


print("evaluator 7개 정의 완료")

# 7. evaluate() 실행 → LangSmith 반영

In [ ]:
from langsmith.evaluation import evaluate

results = evaluate(
    target,
    data=dataset_name,
    evaluators=[
        faithfulness_evaluator,
        answer_relevancy_evaluator,
        context_precision_evaluator,
        context_recall_evaluator,
        answer_correctness_evaluator,
        contains_amount_evaluator,
        no_hallucination_evaluator,
    ],
    experiment_prefix="welfare-ragas-v1"
)

# 8. 결과 시각화

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = results.to_pandas()

metric_cols = [
    "feedback.faithfulness",
    "feedback.answer_relevancy",
    "feedback.context_precision",
    "feedback.context_recall",
    "feedback.answer_correctness",
    "feedback.contains_amount",
    "feedback.no_hallucination",
]

# 실제 존재하는 컬럼만 사용
available = [c for c in metric_cols if c in df.columns]
avg_scores = df[available].mean()
avg_scores.index = [c.replace("feedback.", "") for c in available]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(avg_scores.index, avg_scores.values, color="steelblue")
ax.set_ylim(0, 1.1)
ax.set_title("Welfare RAG Evaluation — Average Scores", fontsize=14)
ax.set_ylabel("Score (0~1)")
ax.axhline(0.8, color="red", linestyle="--", linewidth=1, label="pass threshold (0.8)")
ax.legend()

for bar in bars:
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.02,
        f"{bar.get_height():.2f}",
        ha="center", fontsize=10
    )

plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

print("\n평균 점수:")
print(avg_scores.to_string())